# NeurIPS 2026 Smart Buildings — Version 7

**Experiment: thermodynamic feed-forward dispatch and temperature-headroom-matched airflow.** This is a deliberately *different* control architecture, not a variation of V2/V3 PID feedback, V4 trend forecasting, V5 comfort-regret allocation, or V6 online system identification and candidate search.

## What is new

The AHU mixes return air and outside air before cooling it. Use that physical constraint directly:

- Mixed supply before cooling: T_mix = r * T_return + (1 - r) * T_outside, where r is capped at 0.7.
- Actual delivered temperature: T_delivered = min(T_mix, T_coil_setpoint). The AHU cannot directly heat.
- Each hot zone requests a VAV damper opening in proportion to **comfort excursion / available cooling headroom**. This is a flow *proxy*, not calibrated airflow. A colder supply can reduce damper demand.
- Outside-air mixing is chosen analytically: enable extra free cooling only when the mixed air can meet the coil target **without mechanical cooling**. Otherwise retain 70% recirculation to avoid spending extra exhaust-fan energy. Very cold ambient air is also excluded.
- Reheat and the shared hot-water system are dispatched separately, with outdoor-air compensation because at least 30% of operating air must be outside air.

This has **O(AHUs + zones)** runtime, a fixed small amount of working state, and no nested action search. It is particularly motivated by the recorded **V6 notebook timeout (over nine hours)**.

## Component guide

| Component | Method | Responsibility |
| --- | --- | --- |
| Metadata topology | __init__ | Build dynamic observation and actuator mappings for any building |
| Schedule fallback | _schedule_fallback | Decode time; use comfort or eco band only if live setpoints are invalid |
| Zone state | _read_zone | Read Kelvin temperatures and derive violations |
| Idle actions | _make_idle_actions | Explicitly close VAVs/valves; keep cooling coil and recirculation at idle defaults |
| Thermodynamics | _select_air_supply | Analytic outside-air mixing and cooling-coil setpoint choice |
| Zone air allocation | _control_zone_boxes | Demand / thermal-headroom matching with ventilation floors |
| AHU dispatch | _control_ahu | Coordinate fan pressure from both mean and peak damper demand |
| Hot-water dispatch | _control_hot_water | Outdoor-reset HWS supply, pump on only if a zone needs reheat |
| Execution | get_action | One linear-time action with finite float32/int32 vectors |
| Episode boundary | reset | No persistent learning state to carry across scenarios |

### How to read the implementation

Start with `get_action()`: it orchestrates the five-minute decision. Then read `_control_ahu()` for operating logic and `_select_air_supply()` for the physical temperature calculation. Finally examine `_control_zone_boxes()` and `_control_hot_water()` for per-zone heating and shared plant demand. Methods beginning `_read_`, `_schedule_`, `_clamp`, or `_fraction_action` are input/output helpers, not control algorithms.

### Units, interface and limitations

* Zone air temperatures are **normalized** in the observations and must be denormalized; zone setpoints and outside temperature are **raw Kelvin**.
* AHU coil supply setpoints are in **285–305 K**; normalized continuous action range is **[-1, 1]**. A normalized 0 is a physical midpoint, not off.
* The AHU's free-cooling mixer uses a simple return-air approximation (mean zone temperature) rather than an actual mixed-air sensor. Zone airflow is a heuristic *relative request*, not a calibrated CFM target.
* The 70% return-air cap, 10% VAV damper minimum and 200 Pa running fan minimum follow the official starter-kit physical constraints.
* The return-air temperature is estimated by mean zone temperature, not a measured return sensor. The conservative free-cooling rule intentionally rejects partial chiller savings; energy/carbon trade-offs still need simulator measurement.\n* The rule-based gains are **hypotheses**, not tuned from full reward. This release includes action-contract/synthetic smoke tests only: it does **not** claim a leaderboard improvement.

## Validation and submission

Run the isolated notebook-code contract and latency tests:

    python Co_Kaggle/g8/submissions/test_version7.py

Submit **submission-nb - Version 7.ipynb** as a Kaggle code notebook. Its final cell invokes the official inference server. For a real leaderboard comparison, run the 30 organizer scenarios × 8,640 steps or submit to Kaggle and compare against the **V3 public score 0.06207**. Smaller is better.


In [ ]:
"""Version 7: physics-based, load-matched HVAC dispatch.

Distinct experiment: use the AHU mixing equation and estimated delivered
supply-air temperature to size airflow and avoid unnecessary compressor work.
This is feed-forward engineering control, not PID, temperature prediction,
risk-weighted allocation, RLS, or candidate-action optimization.

Runtime: O(number of AHUs + number of zones) per five-minute step.
"""
import re
import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Weather-compensated, thermodynamics-aware air and hot-water dispatch.

    Action contract: continuous float32 in [-1, 1], discrete int32 in {0, 1}.
    All equipment topology and sensor locations come from episode metadata.
    """

    # Physical constants from the starter-kit action descriptions.
    MIN_SUPPLY_K = 285.0
    MAX_SUPPLY_K = 305.0
    MAX_RETURN_FRACTION = 0.70
    ACTIVE_THRESHOLD_K = 0.18
    MIN_VAV_FRACTION = 0.10
    MIN_FAN_PRESSURE_PA = 200.0
    # A cooling-only AHU still admits at least 30% outside air.
    # Increasing intake beyond that floor also raises exhaust-fan power.
    MIN_ECONOMIZER_OUTDOOR_K = 287.0
    MIN_ECONOMIZER_RECIRC = 0.15

    def __init__(self, building_meta_data, action_spec):
        """Index observations and actions once, never during each time step."""
        super().__init__(building_meta_data, action_spec)
        meta = building_meta_data
        self.n_continuous = int(meta["n_continuous_action"])
        self.n_discrete = int(meta["n_discrete_action"])
        self.fields = {str(s): i for i, s in enumerate(meta["observation_field_names"])}
        self.supply_indices = list(map(int, meta["temp_indices"]))
        self.pressure_indices = list(map(int, meta["pressure_indices"]))
        self.recirc_indices = list(map(int, meta["recirculation_indices"]))
        self.water_indices = list(map(int, meta["water_indices"]))
        self.ahu_count = len(self.supply_indices)
        self.water_supply = self.water_indices[0] if self.water_indices else None
        self.water_pressure = (self.pressure_indices[self.ahu_count]
                               if len(self.pressure_indices) > self.ahu_count else None)
        self.water_run = self.n_discrete - 1

        self.comfort = tuple(map(float, meta["comfort_temp_window"]))
        self.eco = tuple(map(float, meta["eco_temp_window"]))
        self.morning = float(meta["morning_start_hour"])
        self.evening = float(meta["evening_start_hour"])
        normalizer = meta.get("observation_normalization_metadata", {}).get(
            "zone_air_temperature_sensor", {})
        self.zone_mean = float(normalizer.get("sample_mean", 0.0))
        self.zone_std = float(normalizer.get("sample_std", 1.0))
        if not np.isfinite(self.zone_std) or self.zone_std <= 0:
            self.zone_std = 1.0

        def unit_order(key):
            match = re.search(r"\d+", str(key))
            return (int(match.group()) if match else -1, str(key))

        self.ahus = []
        assignments = meta["ahu_zone_assignments"]
        indices = meta["zone_action_indices"]
        for i, identifier in enumerate(sorted(assignments, key=unit_order)[:self.ahu_count]):
            zones = []
            for room in assignments[identifier]:
                vav = str(room).replace("room_", "VAV_")
                temp = self.fields.get(vav + "_zone_air_temperature_sensor")
                if temp is None:
                    continue
                actuators = indices.get(vav, {})
                zones.append({
                    "temperature": temp,
                    "low": self.fields.get(vav + "_zone_air_heating_temperature_setpoint_sensor"),
                    "high": self.fields.get(vav + "_zone_air_cooling_temperature_setpoint_sensor"),
                    "damper": actuators.get("supply_air_damper_percentage_command"),
                    "valve": actuators.get("reheat_valve_percentage_command"),
                })
            self.ahus.append({
                "zones": zones,
                "run": i,
                "supply": self.supply_indices[i],
                "pressure": (self.pressure_indices[i]
                             if i < len(self.pressure_indices) else None),
                "recirc": (self.recirc_indices[i]
                           if i < len(self.recirc_indices) else None),
                "outdoor": self.fields.get(
                    f"ahs_{identifier}_outside_air_temperature_sensor"),
            })
        self.reset()

    def reset(self):
        """No episode state: decisions depend only on the current observation."""
        return None

    @staticmethod
    def _clamp(value, low, high):
        return float(np.clip(value, low, high))

    @classmethod
    def _fraction_action(cls, fraction):
        """Convert physical 0..1 actuator opening to normalized action."""
        return 2.0 * cls._clamp(fraction, 0.0, 1.0) - 1.0

    @staticmethod
    def _temperature_action(kelvin):
        """Supply air 285..305 K maps to -1..1."""
        return float(np.clip((kelvin - 295.0) / 10.0, -1.0, 1.0))

    def _value(self, obs, index):
        """Read an optional sensor without indexing outside the observation."""
        if index is None or not 0 <= index < len(obs):
            return float("nan")
        return float(obs[index])

    def _schedule_fallback(self, obs):
        """Fall back to the occupied/eco schedule if zone setpoints are absent."""
        def clock(sin_name, cos_name, period):
            sine = self._value(obs, self.fields.get(sin_name))
            cosine = self._value(obs, self.fields.get(cos_name))
            if not (np.isfinite(sine) and np.isfinite(cosine)):
                return None
            return float(np.arctan2(sine, cosine) % (2 * np.pi)) * period / (2 * np.pi)

        hour = clock("hod_sin_000", "hod_cos_000", 24.0)
        day = clock("dow_sin_000", "dow_cos_000", 7.0)
        weekday = day is not None and int(day) < 5
        occupied = (weekday and hour is not None
                    and self.morning <= hour < self.evening)
        return self.comfort if occupied else self.eco

    def _read_zone(self, obs, zone, fallback):
        """Decode one zone (Kelvin); return None for an invalid temperature.

        The temperature observation is standardized. Setpoint observations
        are already in Kelvin and take precedence over schedule fallback.
        """
        raw = self._value(obs, zone["temperature"])
        temperature = raw * self.zone_std + self.zone_mean
        if not np.isfinite(temperature) or not 240.0 <= temperature <= 340.0:
            return None
        low = self._value(obs, zone["low"])
        high = self._value(obs, zone["high"])
        if not (np.isfinite(low) and np.isfinite(high)
                and 245.0 <= low < high <= 335.0 and high - low <= 35.0):
            low, high = fallback
        return {
            "actions": zone,
            "temp": temperature,
            "hot": max(0.0, temperature - high),
            "cold": max(0.0, low - temperature),
        }

    def _make_idle_actions(self):
        """A zero action is half-open, so explicitly set idle commands."""
        continuous = np.full(self.n_continuous, -1.0, dtype=np.float32)
        discrete = np.zeros(self.n_discrete, dtype=np.int32)
        for index in self.supply_indices:
            continuous[index] = 1.0  # Warmest: cooling coil effectively idle.
        for index in self.recirc_indices:
            continuous[index] = 1.0  # Max return air, physically capped at 70%.
        return continuous, discrete

    def _select_air_supply(self, zones, outside_k):
        """Select coil setpoint and air mixing from the plant's heat balance.

        Returns (coil_setpoint_K, return_fraction, delivered_K).

        The cooling coil can only LOWER the mixed-air temperature.
        Economizer mode is enabled only if extra outside air can entirely
        avoid mechanical cooling. Otherwise the guaranteed extra exhaust
        airflow can erase a marginal chiller saving; stay at the 70%
        return-air limit until that trade-off has been measured.
        """
        hottest = max(zone["hot"] for zone in zones)
        return_k = float(np.mean([zone["temp"] for zone in zones]))

        if hottest < self.ACTIVE_THRESHOLD_K:
            coil_setpoint_k = self.MAX_SUPPLY_K
        else:
            hottest_zones = [zone["temp"] for zone in zones
                             if zone["hot"] > 0.0]
            mean_hot_k = float(np.mean(hottest_zones))
            coil_setpoint_k = self._clamp(
                mean_hot_k - 4.0 - 1.3 * hottest,
                self.MIN_SUPPLY_K, 297.0)

        if not np.isfinite(outside_k):
            outside_k = return_k

        return_fraction = self.MAX_RETURN_FRACTION
        if (hottest >= self.ACTIVE_THRESHOLD_K
                and self.MIN_ECONOMIZER_OUTDOOR_K <= outside_k < return_k - 1.5):
            temperature_span = return_k - outside_k
            # The lowest mix reachable at 15% return air must be cool
            # enough to meet the coil setpoint without compressor work.
            min_mix_k = (self.MIN_ECONOMIZER_RECIRC * return_k
                         + (1.0 - self.MIN_ECONOMIZER_RECIRC) * outside_k)
            if min_mix_k <= coil_setpoint_k:
                required_return = (coil_setpoint_k - outside_k) / temperature_span
                return_fraction = self._clamp(
                    required_return,
                    self.MIN_ECONOMIZER_RECIRC, self.MAX_RETURN_FRACTION)

        mixed_k = (return_fraction * return_k
                   + (1.0 - return_fraction) * outside_k)
        delivered_k = min(mixed_k, coil_setpoint_k)
        return coil_setpoint_k, return_fraction, delivered_k

    def _control_zone_boxes(self, statuses, delivered, hot_water_demands, continuous):
        """Size VAV flow using actual heating/cooling temperature headroom.

        In contrast to a fixed damper = gain * temperature-error rule,
        requested damper opening falls as cooling headroom grows. This is
        a *proxy* for delivered flow, not a calibrated CFM calculation.
        Returns (mean_damper, peak_damper) for AHU fan coordination.
        """
        sum_damper = 0.0
        peak_damper = 0.0
        for status in statuses:
            zone = status["actions"]
            hot, cold, temperature = (
                status["hot"], status["cold"], status["temp"])
            if hot >= self.ACTIVE_THRESHOLD_K:
                headroom = max(2.5, temperature - delivered)
                damper = self._clamp(0.12 + 2.1 * hot / headroom,
                                     self.MIN_VAV_FRACTION, 0.98)
            elif cold >= self.ACTIVE_THRESHOLD_K:
                # Heating is VAV reheat, never the AHU cooling coil.
                damper = self._clamp(0.22 + 0.21 * cold, 0.18, 0.88)
                if zone["valve"] is not None:
                    valve = self._clamp(0.16 + 0.31 * cold, 0.0, 1.0)
                    continuous[zone["valve"]] = self._fraction_action(valve)
                    hot_water_demands.append((cold, valve))
            else:
                damper = self.MIN_VAV_FRACTION
            sum_damper += damper
            peak_damper = max(peak_damper, damper)
            if zone["damper"] is not None:
                continuous[zone["damper"]] = self._fraction_action(damper)
        # Position fractions are proxies, not calibrated volumetric flow.
        return sum_damper / len(statuses), peak_damper

    def _control_ahu(self, ahu, statuses, outdoor, continuous, discrete, water):
        """Run only if a zone needs conditioning; never call the AHU to heat."""
        if not statuses:
            return
        worst_violation_k = max(max(z["hot"], z["cold"]) for z in statuses)
        if worst_violation_k < self.ACTIVE_THRESHOLD_K:
            return
        target, recirc, delivered = self._select_air_supply(statuses, outdoor)
        mean_damper, peak_damper = self._control_zone_boxes(
            statuses, delivered, water, continuous)
        discrete[ahu["run"]] = 1
        continuous[ahu["supply"]] = self._temperature_action(target)
        if ahu["recirc"] is not None:
            continuous[ahu["recirc"]] = self._fraction_action(recirc)
        if ahu["pressure"] is not None:
            # Use both typical load and worst VAV. A badly overheated room
            # must not disappear in the average of many neutral zones.
            # The fan model uses airflow ~ sqrt(static pressure).
            demand_proxy = max(mean_damper, 0.50 * peak_damper)
            pressure_pa = self.MIN_FAN_PRESSURE_PA + 1250.0 * np.sqrt(
                self._clamp(
                    (demand_proxy - self.MIN_VAV_FRACTION) / 0.65,
                    0.0, 1.0))
            continuous[ahu["pressure"]] = self._clamp(
                pressure_pa / 1000.0 - 1.0, -1.0, 1.0)

    def _control_hot_water(self, water, outdoor_values, continuous, discrete):
        """Weather-compensated hot-water reset, independent of cooling mode.

        Cold outdoor air forces a lower mixed-air temperature because the
        simulator caps recirculation at 70%; compensate with hotter reheat
        water. Only start the pump if a VAV actually requests heating.
        """
        if not water:
            return
        if 0 <= self.water_run < self.n_discrete:
            discrete[self.water_run] = 1
        else:
            return
        worst = max(shortfall for shortfall, _ in water)
        average_valve = sum(valve for _, valve in water) / len(water)
        outside = (sum(outdoor_values) / len(outdoor_values)
                   if outdoor_values else 294.0)
        water_kelvin = self._clamp(
            318.0 + 0.35 * max(0.0, 294.0 - outside) + 5.0 * worst,
            320.0, 349.0)
        if self.water_supply is not None:
            continuous[self.water_supply] = self._fraction_action(
                (water_kelvin - 310.0) / 40.0)
        if self.water_pressure is not None:
            continuous[self.water_pressure] = self._fraction_action(
                self._clamp(0.38 + 0.52 * average_valve, 0.38, 0.95))

    def get_action(self, obs):
        """Read zones -> plan AHU air -> set VAVs -> coordinate shared water.

        Performs one pass per AHU and zone; no search, learned model, or
        state carried between five-minute decisions.
        """
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        continuous, discrete = self._make_idle_actions()
        fallback = self._schedule_fallback(obs)
        hot_water_demands, outdoors = [], []
        for ahu in self.ahus:
            statuses = [
                self._read_zone(obs, z, fallback) for z in ahu["zones"]
            ]
            statuses = [s for s in statuses if s is not None]
            outdoor = self._value(obs, ahu["outdoor"])
            if np.isfinite(outdoor) and 210.0 <= outdoor <= 330.0:
                outdoors.append(outdoor)
            else:
                outdoor = float("nan")
            self._control_ahu(
                ahu, statuses, outdoor, continuous, discrete, hot_water_demands)
        self._control_hot_water(hot_water_demands, outdoors, continuous, discrete)
        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return {"continuous_action": continuous, "discrete_action": discrete}


In [ ]:
# Official Kaggle evaluation gateway. Do not replace with a local CSV writer.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
